# Hotel Price Intelligence
## Guided analysis notebook

Each row is a fictional room offering for one hotel in one month. Hotels can appear in multiple rows. The data is generated from teaching assumptions; findings do not describe actual markets or prove causes. Read, run, inspect, and complete the exercises in order.

## Before you begin
Use Python 3.12 and the project virtual environment described in README.md. From the project root, generate data with python src/generate_data.py, then open this notebook in VS Code and select the project .venv kernel.

In [ ]:
from pathlib import Path
import sys
_cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (_cwd, _cwd.parent)
                     if (p/"src"/"clean_data.py").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open the hotel-price project folder in VS Code.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.clean_data import clean_dataset
sns.set_theme(style="whitegrid")
RAW_PATH = PROJECT_ROOT/"data"/"hotels_raw.csv"
CLEAN_PATH = PROJECT_ROOT/"data"/"hotels_cleaned.csv"
print("Python:", sys.version.split()[0], "| Pandas:", pd.__version__, "| NumPy:", np.__version__)

## Stage 1 ? Python foundations
Variables name values. Common types include text (str), whole numbers (int), decimal numbers (float), and true/false values (bool).

In [ ]:
hotel_name = "Paris Practice Hotel 007"
stars = 4
price_eur = 165.50
has_rating = True
for value in [hotel_name, stars, price_eur, has_rating]:
    print(type(value).__name__, value)

A list holds ordered values; a dictionary holds named values; a loop repeats work; a function packages reusable work.

In [ ]:
prices = [85.0, 130.0, 120.0]
offer = {"city": "Lyon", "price_eur": 85.0}
for price in prices:
    print(f"EUR {price:.2f}")
def after_discount(price, discount):
    return price * (1-discount)
print(offer["city"], after_discount(4*120, .10))

### Your turn
Create a room-size variable and print its type. Loop through three fictional cities. Write a function that adds a 12% service charge. Explain what one dataset row represents.
**Checkpoint:** Why may a hotel ID appear many times?

## Stage 2 ? Load and inspect
Inspect before changing data: shape, types, ranges, missing values, and repeated rows. A raw file is the original generated input.

In [ ]:
if not RAW_PATH.exists():
    raise FileNotFoundError(f"Generate the raw file first with python src/generate_data.py ({RAW_PATH})")
raw = pd.read_csv(RAW_PATH)
print("Rows (offerings):", len(raw), "| columns:", raw.shape[1],
      "| distinct hotels:", raw.hotel_id.nunique())
display(raw.head())

In [ ]:
display(raw.dtypes.rename("dtype").to_frame())
display(raw.describe(include="all").T)
display(raw.isna().sum().rename("missing").to_frame())
print("Exact full-row duplicates:", int(raw.duplicated().sum()))
display(raw.groupby("hotel_id").size().describe().rename("offerings per hotel").to_frame())

A row count means offerings, not hotels. Repeated hotel IDs are expected for different rooms or months. Exact row copies are a separate issue.
### Your turn
Which fields have missing values? Compare offering and hotel counts. Pick a numeric field and describe its range. Predict how many copies were injected. Have raw rows changed?

## Stage 3 ? Clean and validate
Rules: require all 10 fields; validate non-price domains and stable hotel attributes; remove exact full-row duplicates; retain missing ratings and sizes; separate missing, nonnumeric, infinite, zero, or negative prices; retain positive finite prices; flag global IQR outliers but do not delete them.

In [ ]:
cleaned, rejected_prices, audit = clean_dataset(raw)
for key, value in audit.items():
    print(f"{key}: {value}")
print("Rejected price rows:", len(rejected_prices))
display(rejected_prices.head())
display(cleaned.head())

In [ ]:
cleaned.to_csv(CLEAN_PATH, index=False)
print("Saved separately:", CLEAN_PATH)
print("Reconciles:",
      audit["input_rows"]-audit["duplicate_copies_removed"]
      == audit["retained_rows"]+audit["invalid_price_rows_separated"])
print("Raw rows still on disk:", len(pd.read_csv(RAW_PATH)))

Exact duplicates can give one offering extra weight. Missing ratings or sizes do not prevent other price analyses. Invalid prices cannot support price summaries. A valid but extreme price is flagged, not assumed wrong.
### Your turn
Verify the audit equation. Which missing values remain? Distinguish an invalid price from an unusual valid price.

## Stage 4 ? Compare prices and visualize
Report counts because groups can differ in size. Means respond to extremes; medians are middle values and often less sensitive. Interpret both with distributions.

In [ ]:
def price_summary(data, group=None):
    if group:
        return data.groupby(group)["price_eur"].agg(
            count="count", mean="mean", median="median", std="std").reset_index()
    return pd.DataFrame([{"count":data.price_eur.count(),
        "mean":data.price_eur.mean(),"median":data.price_eur.median(),
        "std":data.price_eur.std()}])
display(price_summary(cleaned).round(2))
for col in ["city","star_rating","month"]:
    print("By", col, "(count is offerings)")
    display(price_summary(cleaned,col).round(2))

### Chart 1 ? Valid nightly price distribution. Population: all retained positive, finite prices. Bars count offerings.

In [ ]:
if cleaned.empty: print("No valid prices to plot.")
else:
    fig,ax=plt.subplots(figsize=(9,5))
    sns.histplot(data=cleaned,x="price_eur",bins=30,ax=ax)
    ax.set(title="Retained nightly price distribution",xlabel="Nightly price (EUR)",ylabel="Offerings")
    ax.text(.99,.97,f"Population: {len(cleaned)} offerings",ha="right",va="top",transform=ax.transAxes)
    plt.tight_layout(); plt.show()

### Chart 2 ? City comparison. Boxes show retained offering distributions; labels show offering counts. These are synthetic offers, not real city prices.

In [ ]:
if cleaned.empty: print("No valid prices for city comparison.")
else:
    counts=cleaned.groupby("city").price_eur.count()
    order=cleaned.groupby("city").price_eur.median().sort_values().index
    fig,ax=plt.subplots(figsize=(10,5))
    sns.boxplot(data=cleaned,x="city",y="price_eur",order=order,ax=ax)
    ax.set_xticklabels([f"{c}
(n={counts[c]})" for c in order])
    ax.set(title="Nightly prices by city",xlabel="City (n = offerings)",ylabel="Nightly price (EUR)")
    plt.tight_layout(); plt.show()

### Chart 3 ? Star-classification comparison. Stars are not guest ratings; each point is an offering, so hotels can repeat.

In [ ]:
if cleaned.empty: print("No valid prices for star comparison.")
else:
    counts=cleaned.groupby("star_rating").price_eur.count()
    order=sorted(cleaned.star_rating.unique())
    fig,ax=plt.subplots(figsize=(9,5))
    sns.boxplot(data=cleaned,x="star_rating",y="price_eur",order=order,ax=ax)
    ax.set_xticklabels([f"{s} star(s)
(n={counts[s]})" for s in order])
    ax.set(title="Nightly prices by star classification",
           xlabel="Star classification (n = offerings)",ylabel="Nightly price (EUR)")
    plt.tight_layout(); plt.show()

### Your turn
Compare city means and medians. Describe the histogram. Do star groups overlap? Give one cautious finding with a count.

## Stage 5 ? Explore relationships
Pearson correlation describes linear association from -1 to +1; it does not prove causation. Use complete pairs and report their count. Hotel IDs are identifiers, not measurements.

In [ ]:
def pearson_result(data,x,y):
    pairs=data[[x,y]].dropna(); n=len(pairs)
    if n<2 or pairs[x].nunique()<2 or pairs[y].nunique()<2:
        return {"x":x,"y":y,"usable_pairs":n,"pearson_r":None}
    return {"x":x,"y":y,"usable_pairs":n,
            "pearson_r":float(pairs[x].corr(pairs[y]))}
display(pd.DataFrame([
    pearson_result(cleaned,"room_size_m2","price_eur"),
    pearson_result(cleaned,"guest_rating","price_eur")]))

### Chart 4 ? Room size and price. Population: offerings with both values. Room type, city, stars, and month may influence both.

In [ ]:
pairs=cleaned[["room_size_m2","price_eur","city"]].dropna()
if len(pairs)<2: print("Not enough complete pairs.")
else:
    fig,ax=plt.subplots(figsize=(9,6))
    sns.scatterplot(data=pairs,x="room_size_m2",y="price_eur",hue="city",alpha=.75,ax=ax)
    ax.set(title="Room size and nightly price",xlabel="Room size (square metres)",ylabel="Nightly price (EUR)")
    ax.text(.99,.02,f"{len(pairs)} complete offering pairs",ha="right",transform=ax.transAxes)
    plt.tight_layout(); plt.show()

### Outlier sensitivity. Compare summaries and room-size/price correlation with all valid prices and without flagged prices. The latter is a sensitivity check.

In [ ]:
without_outliers=cleaned.loc[~cleaned.price_outlier]
rows=[]
for label,data in [("All valid prices",cleaned),("Excluding flagged prices",without_outliers)]:
    rows.append({"population":label,"offerings":len(data),
      "mean_eur":data.price_eur.mean(),"median_eur":data.price_eur.median(),
      "room_size_price_pairs":len(data[["room_size_m2","price_eur"]].dropna()),
      "room_size_price_r":pearson_result(data,"room_size_m2","price_eur")["pearson_r"]})
display(pd.DataFrame(rows).round(3))

### Optional peer screen. A heuristic, not a value ranking: compare within city, stars, room type, and month; require 5 offerings, rating at least 8, and price below group median. It omits amenities and booking conditions.

In [ ]:
keys=["city","star_rating","room_type","month"]
peers=cleaned.dropna(subset=["guest_rating"]).copy()
if peers.empty: print("Insufficient evidence: no rated offerings.")
else:
    groups=peers.groupby(keys)["price_eur"]
    peers["peer_count"]=groups.transform("size")
    peers["peer_median_eur"]=groups.transform("median")
    candidates=peers[(peers.peer_count>=5)&(peers.guest_rating>=8)&
                     (peers.price_eur<peers.peer_median_eur)]
    if candidates.empty: print("No offerings met the heuristic; report insufficient evidence.")
    else: display(candidates[keys+["hotel_id","guest_rating","price_eur","peer_count","peer_median_eur"]].head(20))

### Your turn
Report room-size correlation and pair count. Compare the outlier-excluded result. Why does near-zero not prove no relationship? What could confound room size and price?

## Stage 6 ? Findings and reproducibility

Write three findings. State each question, population and count, calculated evidence with units, interpretation, and limitations. Do not claim causality or real-market facts.

### Finding 1 ? City, stars, or month
Question / population and count / evidence / interpretation / limitation:

### Finding 2 ? Room size or guest rating
Question / usable-pair count / evidence / interpretation / limitation:

### Finding 3 ? Sensitivity to unusual prices
Evidence with and without flagged prices / interpretation / limitation:

### Demonstration outline
Explain one row and the fictional data. Show raw quality, one cleaning rule and reconciled counts, a group comparison, a relationship chart, outlier sensitivity, and three findings.

### Final checkpoint
Restart the kernel and run every cell top to bottom. Confirm the raw input remains unchanged, cleaned data is recreated, counts reconcile, four charts render, and statistics match their populations. Assumptions are synthetic; rows repeat hotels; missingness reduces some populations; global fences ignore peer differences; taxes, availability, and booking terms are omitted.